## Training-Datensatz Generierung

Dieses Script wird genutzt, um mittels eines bestehenden Corpus und einem Prompt mit 5 manuell erstellten Beispiel-Fragen einen LLM-generierten Trainings-Datensatz zu erzeugen.

Der resultierende Datensatz wird eine Liste von single positive Query-Chunk Paaren enthalten.

In [ ]:
import os
import random
import json

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.exceptions import OutputParserException

from datasets import load_dataset
from tqdm import tqdm

## API-Key hinterlegen

Um GPT-5 für die Generierung der Trainingsfragen zu nutzen wird ein API-Key benötigt.

In [ ]:
os.environ["OPENAI_API_KEY"] = "Hier eigenen OPENAI_API_KEY einfügen"

## Test Parameter

Folgende Code-Zelle enthält alle anzupassenden Parameter für die Generierung eines neuen Trainings-Datensatzes.

Um sicherzustellen, dass die Fragen gleichmäig über die vorhanden Chunks aufgeteilt sind, wird nicht mit einem Absolutwert an Fragen, sondern mit einer definierten Menge an Fragen pro Chunk gearbeitet.

In [ ]:
# Pfad des zu verwendenden Corpus.
# Datensätze müssen den Vorgaben des SentenceTransformers entsprechen.
corpus_location = "datasets/GerManualDPR/full_corpus.json"

# Menge der zu erstellenden Fragen pro Chunk
queries_per_chunk = 3

# Parameter für die stichprobenartige Generierung von Fragen zu x zufällig ausgewählten Chunks, um potenzielle 
# Qualitätsmängel in den generierten Fragen zu identifizieren und den Prompt entsprechend anzupassen.
test_run = False;
SEED = 314;
test_chunk_amount = 2;

# Pfad der resultierenden JSON Datei
out_path = "datasets/GerManualDPR/unfiltered_synthetic/train_dataset.json"

# Prompt für die Erzeugung der Fragen.
label_template = """
Du bist eine technische Assistenz für Industrie- und Anlagenbau. Deine Aufgabe ist es, präzise, faktenbasierte Fragen zu generieren, die realistisch von Fachkräften aus Inbetriebnahme, Aufbau von Produktionsanlagen und Betrieb gestellt würden.

Gegeben:
{chunk}  // JSON- oder Textobjekt des Abschnitts; verwende NUR diesen Inhalt

Parameter:
- Maximale Anzahl Fragen: {n}

Leitfaden (unbedingt einhalten):
1) Bodenhaftung: Jede Frage MUSS ausschließlich aus dem gegebenen Abschnitt beantwortbar sein – ohne externes Wissen oder Annahmen.
2) Entitätsbezug: Verknüpfe jede Frage klar mit einer konkreten Entität aus dem Abschnitt (z. B. Komponente/Bauteil, Parameter, Menüpunkt, Fehlercode, Werkzeug, Sicherheitshinweis, Grenzwert, Schritt im Ablauf).
3) Zweck: Frage nach überprüfbaren Fakten, klaren Anweisungen, Bedingungen, Grenzwerten, Abhängigkeiten, Reihenfolgen, Parametereinstellungen, Ursachen/Wirkungen, Sicherheitsanforderungen.
4) Paraphrase statt Zitat: Übernimm KEINE Sätze wortgleich aus dem Abschnitt. Formuliere um (Synonyme, andere Reihenfolge), ohne Inhalt zu verändern.
5) Abdeckung: Wenn mehrere Fragen erzeugt werden, sollen sie unterschiedliche relevante Teile/Aspekte des Abschnitts abdecken (nicht alle zum gleichen Satz/Parameter).
6) Natürlichkeit & Präzision: Kurze, präzise, natürlich klingende Fragen aus der Praxis; keine Füllwörter, keine Rhetorik, keine Anreden.
7) Grammatik/Orthografie: Leichte Fehler sind erlaubt (realistische Eingaben), ABER die Frage muss ohne Interpretation eindeutig verständlich bleiben.
8) Kein Inhalt → Keine Frage: Enthält der Abschnitt hauptsächlich unbrauchbaren Text (z. B. reine Listen ohne erklärenden Kontext, Abbildungen/Tabellen-Platzhalter, stark fehlerhafte Konvertierung), dann gib KEINE Fragen zurück.
9) Keine Erfindungen: Erfinde keine Entitäten, Werte, Schritte oder Bedingungen, die nicht im Abschnitt stehen.
10) Verwende die Bezeichnung des übergeordneten Geräts oder Systems, sofern sie im Präfix des Abschnitts enthalten ist, in jeder Frage und binde sie möglichst natürlich in den Fließtext ein.

Stilhinweise:
- bevorzuge W-Fragen (Was/Wie/Welcher/Welche/Welches/Wann/Wo/Unter welcher Bedingung…)
- spezifisch und unmissverständlich (lieber „Welcher Parameter X muss auf Y gesetzt werden, um Z zu aktivieren?“ als „Wie einstellen?“)
- Einheiten, Grenzwerte, Zustände exakt erfragen, falls im Abschnitt vorhanden

Ausgabeformat (VALIDES JSON):
- Gib ein JSON-Objekt zurück mit Schlüsseln "question_1" bis "question_{n}", wobei m = Anzahl erzeugter Fragen (1…{n}).
- Erzeuge KEINE zusätzlichen Felder.
- Wenn keine geeignete Frage möglich ist, gib ein leeres JSON-Objekt zurück.

Beispiele für gewünschte Fragestile (nur als Stilreferenz, NICHT übernehmen):
- „Es soll die Wichtungsart der Drehmomentdaten unseres ctrlX Drive Antriebs angepasst werden - Über welchen Parameter kann diese Änderung vorgenommen werden?“
- „Welche Armoberflächentemperatur kann ein Stäubli TX2 40 erreichen? Stellt das eine Gefahr für den Nutzer dar?“
- „In welchem Parameter meiner ctrlX DRIVE Steuerung ist der aktuelle Vorgabewert der Gerschwindigkeit zu sehen?"
- „Welche Parameter können bei der ctrlx Drive Betriebsart "Cyclic sync torque mode" zur Anpassung des Sollwert-Filters genutzt werden?“
- „Welche Konfigurationsmöglichkeiten bietet das ctrlX Drive Engineering Tool für die Messtasterfunktionalität?"

Jetzt generiere bis zu {n} Fragen auf Deutsch zum gegebenen Abschnitt.
"""

## Generierung des vollständigen Test-Datensatzes

Nun wird der Corpus geladen und GPT-5 mit der von OpenAI fest vorgegebenen temperature von 1 als LLM-Modell für die Fragen-Generierung gesetzt:

In [ ]:
corpus_dataset = load_dataset("json", data_files=corpus_location, split="train")

if test_run:
    random.seed(SEED)
    # wähle zufällige Indizes und selektiere daraus ein reduziertes Dataset
    idxs = random.sample(range(len(corpus_dataset)), k=test_chunk_amount)
    corpus_dataset = corpus_dataset.select(idxs)

chunk_texts_src = corpus_dataset["positive"]
chunk_ids_src = corpus_dataset["chunk_id"]

# Erzeugen eines Prompt-Objekts aus dem Template. Dies ermöglicht das dynamische ersetzen des {chunk}-Platzhalters
label_prompt = ChatPromptTemplate.from_template(label_template)

# Definieren des zu verwendenden LLM-Modells
llm = ChatOpenAI(temperature=1, model="gpt-5")

# Fordert stets JSON-Objekte von der API
llm_json = llm.bind(response_format={"type": "json_object"})

# Korrektes JSON-Format in Antwort mittels JsonOutputParser() sicherstellen
label_chain = label_prompt | llm_json | JsonOutputParser()

# Generation mit Inline-Extraktion & Retries um sicherzustellen, dass die n Fragen zu jedem Chunk generiert werden ---
anchors = []       # flache Liste aller Fragen
positives   = []   # korrespondierende Anchor-Einträge
chunk_ids = []     # korrespondierende Chunk-ID

for idx, (cid, ch) in enumerate(tqdm(list(zip(chunk_ids_src, chunk_texts_src)),
                                     desc="Generating questions",
                                     total=len(chunk_texts_src))):
    for attempt in range(10):
        try:
            out = label_chain.invoke({"chunk": ch, "n": queries_per_chunk})

            questions = [out[f"question_{i}"] for i in range(1, queries_per_chunk + 1)]
            if not all(isinstance(q, str) and q.strip() for q in questions):
                raise ValueError("Empty or non-string question detected")

            anchors.extend(questions)
            positives.extend([ch] * queries_per_chunk)
            chunk_ids.extend([cid] * queries_per_chunk)
            break

        except (OutputParserException, KeyError, ValueError) as e:
            if attempt < 9:
                continue
            print(f"Warn: Chunk #{idx} konnte nach 10 Versuchen nicht generiert werden und wurde übersprungen. Grund: {e}")


# Überprüfen ob die Menge der Positives mit der Menge der Anchors übereinstimmt
if len(anchors) != len(positives):
    print (f'Es wurden insgesamt {len(anchors)} Fragen generiert. Es hätten jedoch {len(positives)} Fragen generiert werden sollen')
else:
    print ("Generierung war erfolgreich!")

records = []
for i in range(len(anchors)):
    records.append({
        "positive": positives[i],
        "anchor": f"query: {anchors[i]}",
        "id": i + 10000,                 # keine Überschneidungen mit IDs aus Test-Datensatz
        "chunk_id": int(chunk_ids[i])
    })

os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

print("✓ saved JSON list:", out_path, "| rows:", len(records))